# Semantic Kernel 

এই কোড নমুনায়, আপনি একটি সহজ এজেন্ট তৈরি করতে [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI ফ্রেমওয়ার্ক ব্যবহার করবেন।

এই নমুনার উদ্দেশ্য হল সেই ধাপগুলি প্রদর্শন করা যা পরবর্তীতে বিভিন্ন এজেন্টিক প্যাটার্ন বাস্তবায়নের সময় অতিরিক্ত কোড নমুনায় প্রয়োগ করা হবে।


## প্রয়োজনীয় পাইথন প্যাকেজগুলি আমদানি করুন


In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## ক্লায়েন্ট তৈরি করা

এই উদাহরণে, আমরা LLM অ্যাক্সেস করার জন্য [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) ব্যবহার করব।

`ai_model_id` সেট করা হয়েছে `gpt-4o-mini` এ। বিভিন্ন ফলাফল লক্ষ্য করার জন্য GitHub Models মার্কেটপ্লেসে উপলব্ধ অন্য মডেলে পরিবর্তন করে দেখুন।

GitHub Models এর জন্য প্রয়োজনীয় `base_url` এর জন্য `Azure Inference SDK` ব্যবহারের জন্য, আমরা Semantic Kernel-এর মধ্যে `OpenAIChatCompletion` কানেক্টর ব্যবহার করব। এছাড়াও অন্যান্য [উপলব্ধ কানেক্টর](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) রয়েছে যেগুলি Semantic Kernel কে অন্যান্য মডেল প্রদানকারীর সাথে ব্যবহার করার সুযোগ দেয়।


In [ ]:
import random   

# Define a sample plugin for the sample

class DestinationsPlugin:
    """A List of Random Destinations for a vacation."""

    def __init__(self):
        # List of vacation destinations
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # Track last destination to avoid repeats
        self.last_destination = None

    @kernel_function(description="Provides a random vacation destination.")
    def get_random_destination(self) -> Annotated[str, "Returns a random vacation destination."]:
        # Get available destinations (excluding last one if possible)
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # Select a random destination
        destination = random.choice(available_destinations)

        # Update the last destination
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# Create an AI Service that will be used by the `ChatCompletionAgent`
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## এজেন্ট তৈরি করা

এখানে, আমরা `TravelAgent` নামক এজেন্ট তৈরি করছি।

এই উদাহরণে, আমরা খুবই মৌলিক নির্দেশাবলী ব্যবহার করেছি। এজেন্টের আচরণ কীভাবে পরিবর্তিত হয় তা দেখতে এই নির্দেশাবলী পরিবর্তন করতে মুক্ত মনে করুন।


In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
)

## এজেন্টগুলি চালানো

এখন আমরা `ChatHistory` সেট আপ করে এবং এর মধ্যে `system_message` অন্তর্ভুক্ত করে এজেন্টটি চালাতে পারি। আমরা পূর্বে প্রতিষ্ঠিত `AGENT_INSTRUCTIONS` ব্যবহার করব।

এগুলি সেট আপ করার পর, আমরা `user_inputs` তৈরি করি, যা নির্দেশ করে ব্যবহারকারী এজেন্টকে কী পাঠাচ্ছেন। এই উদাহরণে, বার্তাটি সেট করা হয়েছে `Plan me a sunny vacation`।

আপনি এজেন্টটি কিভাবে আলাদাভাবে প্রতিক্রিয়া জানায় দেখতে এই বার্তাটি পরিবর্তন করতে পারেন।


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # Buffer to reconstruct streaming function call
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # Accumulate arguments (streamed in chunks)
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # Finalize any pending function call before showing result
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # leave as raw string

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**দস্তাবেজ স্বীকারোক্তি**:
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতার জন্য চেষ্টা করি, তবে জেনে রাখুন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার নিজ মাতৃভাষায় প্রামাণিক উৎস হিসেবে গণ্য করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহার থেকে উদ্ভূত যে কোনো ভুল বুঝাবুঝি বা ভ্রান্ত ব্যাখ্যার জন্য আমরা দায়ী নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
